# TrueRender v3 — Alpha-Mask Reconstruction Pipeline

Goal: phone video -> object-only 3D mesh (`GLB`, `STL`, `OBJ`) with a cleaner mesh-extraction path than v2.

The main v2 failure was not detection or 2DGS training quality; it was that SAM masks were baked into white-background JPGs. v3 keeps SAM masks as real alpha masks and trains 2DGS on RGBA PNGs so `gt_alpha_mask` can be used during TSDF fusion.

Pipeline:

1. Extract sharp frames from `hydroflaskgreen.MOV`
2. Detect the target object with Grounding DINO
3. Segment with SAM 3 and save both binary masks and RGBA training images
4. Estimate poses with VGGT on original RGB frames
5. Build a COLMAP-style dataset whose image names point to RGBA PNGs
6. Train 2DGS
7. Run TSDF mesh extraction with real alpha masks
8. Export final mesh, with explicit fallback to the known clean v1 SuGaR mesh

In [1]:
# Global configuration
import os
import glob
import json
import time
import shutil
import subprocess
from pathlib import Path

import numpy as np
from PIL import Image

PROJECT_NAME = "truerender_v3"
DRIVE_ROOT = "/content/drive/MyDrive"

VIDEO_PATH = f"{DRIVE_ROOT}/hydroflaskgreen.MOV"

FRAMES_RAW = "/content/frames_raw_v3"
FRAMES_KEEP = "/content/frames_v3"
SAM_INPUT = "/content/sam3_input_v3"
MASKS_DIR = "/content/masks_v3"
FRAMES_RGBA = "/content/frames_rgba_v3"
COLMAP_OUT = "/content/colmap_vggt_v3"
OUTPUT_2DGS = "/content/output_2dgs_v3"
OUTPUT_DIR = "/content/outputs_v3"

DRIVE_FRAMES = f"{DRIVE_ROOT}/truerender_frames_v3"
DRIVE_MASKS = f"{DRIVE_ROOT}/truerender_masks_v3"
DRIVE_RGBA = f"{DRIVE_ROOT}/truerender_rgba_frames_v3"
DRIVE_COLMAP = f"{DRIVE_ROOT}/truerender_colmap_v3"
DRIVE_2DGS = f"{DRIVE_ROOT}/truerender_2dgs_output_v3"
DRIVE_FINAL = f"{DRIVE_ROOT}/truerender_final_v3"

N_KEEP = 30
FPS = 5
USE_CHECKPOINTS = True
TRAIN_ITERS = 7000

for d in [FRAMES_RAW, FRAMES_KEEP, SAM_INPUT, MASKS_DIR, FRAMES_RGBA, COLMAP_OUT, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("TrueRender v3 config ready")

TrueRender v3 config ready


In [3]:
from google.colab import drive

drive.mount("/content/drive")

# Optional: authenticate Hugging Face Hub downloads (higher rate limits, fewer warnings)
# Colab: add a secret named HF_TOKEN (Settings -> Secrets) with a read token from huggingface.co/settings/tokens
try:
    from google.colab import userdata

    _hf = userdata.get("HF_TOKEN")
    if _hf:
        os.environ["HF_TOKEN"] = _hf
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf
        print("HF_TOKEN loaded from Colab secrets")
except Exception:
    # Not on Colab, or secrets not configured — public downloads still work, just with lower quotas.
    pass


def reset_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def copytree_fresh(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


def restore_checkpoint(src, dst, label):
    if USE_CHECKPOINTS and os.path.exists(src):
        reset_dir(dst)
        for item in sorted(os.listdir(src)):
            s = os.path.join(src, item)
            d = os.path.join(dst, item)
            if os.path.isdir(s):
                shutil.copytree(s, d)
            else:
                shutil.copy2(s, d)
        print(f"Restored {label}: {len(os.listdir(dst))} files")
        return True
    return False


def assert_nonempty_dir(path, label):
    files = sorted(os.listdir(path)) if os.path.exists(path) else []
    assert files, f"{label} missing or empty: {path}"
    return files

print("Drive and helpers ready")

Mounted at /content/drive
Drive and helpers ready


### Optional — Hugging Face Token In Notebook

Run the next cell if you want authenticated Hugging Face downloads without using Colab Secrets. The token is entered with `getpass`, stored only in this runtime's environment, and is not saved into the notebook.

In [5]:
# Optional: paste your Hugging Face read token here at runtime.
# Leave blank and press Enter to continue anonymously.
import getpass

if not os.environ.get("HF_TOKEN"):
    hf_token = getpass.getpass("HF_TOKEN (optional, input hidden): ").strip()
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN set for this runtime")
    else:
        print("Continuing without HF_TOKEN")
else:
    print("HF_TOKEN already set for this runtime")

HF_TOKEN set for this runtime


## Stage 1 — Frame Extraction

Extract frames from the phone video, score them by Laplacian sharpness, keep the sharpest `N_KEEP`, then restore chronological order. These original RGB frames are preserved for VGGT pose estimation.

In [6]:
import cv2

if not restore_checkpoint(DRIVE_FRAMES, FRAMES_KEEP, "filtered frames"):
    assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
    reset_dir(FRAMES_RAW)
    reset_dir(FRAMES_KEEP)

    t0 = time.time()
    subprocess.run([
        "ffmpeg", "-y", "-i", VIDEO_PATH,
        "-vf", f"fps={FPS}",
        f"{FRAMES_RAW}/frame_%05d.jpg",
    ], check=True)

    scored = []
    for path in sorted(glob.glob(f"{FRAMES_RAW}/*.jpg")):
        gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        score = cv2.Laplacian(gray, cv2.CV_64F).var()
        scored.append((path, score))

    keep = sorted([p for p, _ in sorted(scored, key=lambda x: x[1], reverse=True)[:N_KEEP]])
    for dst_idx, src in enumerate(keep):
        shutil.copy2(src, f"{FRAMES_KEEP}/frame_{dst_idx:05d}.jpg")

    copytree_fresh(FRAMES_KEEP, DRIVE_FRAMES)
    print(f"Kept {len(keep)} / {len(scored)} frames in {time.time() - t0:.1f}s")

frames = assert_nonempty_dir(FRAMES_KEEP, "filtered frames")
print(frames[:3], "...", frames[-3:])

Restored filtered frames: 30 files
['frame_00000.jpg', 'frame_00001.jpg', 'frame_00002.jpg'] ... ['frame_00027.jpg', 'frame_00028.jpg', 'frame_00029.jpg']


## Stage 2 — Object Detection

Use Grounding DINO on the first filtered frame to pick the target object. The detected phrase is enriched before being passed to SAM 3, because SAM text prompts usually need more concrete object wording than detector labels.

### Colab note — Grounding DINO + Transformers

If you see `BertModel ... get_head_mask`, your environment is on **Transformers 5.x**. v3 pins **Transformers 4.41.2** for compatibility.

If the pin cell prints a non-4.x version, use **Runtime → Restart session**, then rerun from the configuration cell so nothing imports `transformers` before the pin installs.

In [7]:
# Grounding DINO setup and first-frame detection
#
# Colab often ships Transformers 5.x, which removes BertModel.get_head_mask and breaks groundingdino-py.
# We do three things:
# 1) force-install a compatible Transformers 4.x (pin hard; other packages may try to upgrade you back to 5.x)
# 2) verify the *imported* transformers matches that version (catches stale in-memory imports)
# 3) patch groundingdino's bertwarper.py for Transformers 5.x if needed (belt + suspenders)

import sys


def _pip_show_version(pkg: str) -> str:
    p = subprocess.run(["python", "-m", "pip", "show", pkg], capture_output=True, text=True, check=False)
    for line in (p.stdout or "").splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    return ""


def _imported_transformers_probe() -> tuple[str, str, str]:
    # Run in a fresh interpreter to avoid stale sys.modules in the notebook kernel.
    cmd = (
        "import transformers\n"
        "from transformers import BertModel\n"
        "print(transformers.__version__)\n"
        "print(transformers.__file__)\n"
        "print(int(hasattr(BertModel, 'get_head_mask')))\n"
    )
    p = subprocess.run(["python", "-c", cmd], capture_output=True, text=True, check=False)
    out = (p.stdout or "").strip().splitlines()
    if len(out) < 3 or p.returncode != 0:
        raise RuntimeError(
            "Failed to probe transformers in a clean interpreter.\n"
            f"returncode={p.returncode}\nstdout:\n{p.stdout}\nstderr:\n{p.stderr}"
        )
    ver, path, has_attr = out[0], out[1], out[2]
    return ver, path, has_attr


def _pin_transformers_441() -> None:
    # Install 4.41.2 without letting pip "upgrade you back" for unrelated deps.
    subprocess.run(
        [
            "python",
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--force-reinstall",
            "--no-deps",
            "transformers==4.41.2",
        ],
        check=True,
    )
    subprocess.run(
        [
            "python",
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "tokenizers>=0.14,<0.20",
            "huggingface-hub<0.24",
            "safetensors",
        ],
        check=True,
    )


def _install_groundingdino_py() -> None:
    subprocess.run(
        ["python", "-m", "pip", "install", "-q", "--upgrade", "groundingdino-py"],
        check=True,
    )


def _ensure_transformers_441_stack() -> None:
    _pin_transformers_441()
    _install_groundingdino_py()
    # groundingdino-py install can pull Transformers 5.x as a dependency — re-pin after.
    if not _pip_show_version("transformers").startswith("4."):
        print("Re-pinning Transformers 4.41.2 after groundingdino-py install...")
        _pin_transformers_441()


_ensure_transformers_441_stack()


pip_ver = _pip_show_version("transformers")
print(f"pip reports transformers=={pip_ver}")

imp_ver, imp_path, has_attr = _imported_transformers_probe()
print(f"python -c import reports transformers=={imp_ver}")
print(f"transformers file: {imp_path}")
print(f"BertModel has get_head_mask (type-level): {has_attr}")

# If the notebook kernel already imported Transformers 5.x earlier in the session, metadata can look
# fine while the kernel is still "stuck" — restarting fixes that. We at least detect the common failure mode.
if "transformers" in sys.modules:
    print("NOTE: `transformers` is already imported in this notebook kernel.")
    print("If anything below looks inconsistent, use Runtime → Restart session and rerun from the top.")

if not pip_ver.startswith("4."):
    raise RuntimeError(
        "pip did not end up on Transformers 4.x. "
        "Restart the Colab runtime and rerun this notebook from the first cell."
    )

if not imp_ver.startswith("4."):
    raise RuntimeError(
        f"Your environment is still resolving Transformers {imp_ver} for `python -c import transformers`. "
        "Restart the Colab runtime (this is usually caused by a partially-upgraded site-packages state)."
    )


def patch_groundingdino_bertwarper() -> None:
    """Patch site-packages groundingdino bertwarper for Transformers 5.x API drift."""
    cmd = r"""import pathlib
import groundingdino
p = pathlib.Path(groundingdino.__file__).parent / "models" / "GroundingDINO" / "bertwarper.py"
text = p.read_text()
if "TRUERENDER_PATCH_BERTWARPER_V1" in text:
    print("bertwarper.py already patched")
    raise SystemExit(0)

old = "        self.get_head_mask = bert_model.get_head_mask\n"
new = (
    "        # TRUERENDER_PATCH_BERTWARPER_V1: Transformers>=5 removed BertModel.get_head_mask\n"
    "        self.get_head_mask = getattr(bert_model, \"get_head_mask\", None)\n"
)
if old not in text:
    raise SystemExit(f\"Could not find expected line in {p}\\n\" + \"\\n\".join(text.splitlines()[20:40]))

old_call = "        head_mask = self.get_head_mask(head_mask, self.config.num_hidden_layers)\n"
new_call = (
    "        if self.get_head_mask is None:\n"
    "            head_mask = [None] * self.config.num_hidden_layers\n"
    "        else:\n"
    "            head_mask = self.get_head_mask(head_mask, self.config.num_hidden_layers)\n"
)
if old_call not in text:
    raise SystemExit(f\"Could not find head_mask call in {p}\\n\" + \"\\n\".join(text.splitlines()[80:110]))

text = text.replace(old, new, 1).replace(old_call, new_call, 1)
p.write_text(text)
print(f\"patched: {p}\")
"""
    p = subprocess.run(["python", "-c", cmd], capture_output=True, text=True, check=False)
    print((p.stdout or "").strip())
    if p.returncode != 0:
        raise RuntimeError(
            "Failed patching groundingdino bertwarper.py.\n"
            f"returncode={p.returncode}\nstdout:\n{p.stdout}\nstderr:\n{p.stderr}"
        )


if has_attr == "0":
    print("Transformers removed type-level get_head_mask; patching groundingdino bertwarper.py...")
    patch_groundingdino_bertwarper()

import torch
from groundingdino.util.inference import load_model, load_image, predict

DINO_WEIGHTS = "/content/groundingdino_swint_ogc.pth"
DINO_URLS = [
    # Hugging Face mirror (more reliable than GitHub releases in some Colab sessions)
    "https://huggingface.co/ShilongLiu/GroundingDINO/resolve/main/groundingdino_swint_ogc.pth",
    # GitHub release (original)
    "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha2/groundingdino_swint_ogc.pth",
]


def _download_to_file(url: str, dst: str, min_bytes: int) -> None:
    import urllib.request

    tmp = dst + ".partial"
    if os.path.exists(tmp):
        os.remove(tmp)

    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (Colab) TrueRender/3.0",
            "Accept": "*/*",
        },
        method="GET",
    )

    with urllib.request.urlopen(req) as resp, open(tmp, "wb") as out:
        if resp.status != 200:
            raise RuntimeError(f"HTTP {resp.status} for {url}")
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)

    size = os.path.getsize(tmp)
    if size < min_bytes:
        os.remove(tmp)
        raise RuntimeError(f"Download too small ({size} bytes) from {url}")

    os.replace(tmp, dst)


def ensure_dino_weights(path: str, urls, min_mb: float = 150.0) -> None:
    min_bytes = int(min_mb * 1e6)
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return

    if os.path.exists(path):
        os.remove(path)

    last_err = None
    for url in urls:
        try:
            print(f"Downloading Grounding DINO weights from:\n  {url}")
            _download_to_file(url, path, min_bytes=min_bytes)
            print(f"Saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)")
            return
        except Exception as exc:
            last_err = exc
            if os.path.exists(path):
                os.remove(path)
            partial = path + ".partial"
            if os.path.exists(partial):
                os.remove(partial)
            print(f"  failed: {exc}")

    raise RuntimeError(
        "Could not download Grounding DINO weights. "
        "Try again later, upload the .pth to Drive manually, or set DINO_WEIGHTS to a local path."
    ) from last_err


ensure_dino_weights(DINO_WEIGHTS, DINO_URLS)

import groundingdino
DINO_CONFIG = os.path.join(os.path.dirname(groundingdino.__file__), "config", "GroundingDINO_SwinT_OGC.py")
dino = load_model(DINO_CONFIG, DINO_WEIGHTS)

FIRST_FRAME = sorted(glob.glob(f"{FRAMES_KEEP}/*.jpg"))[0]
DINO_CAPTION = "object . bottle . container . cup . item . product"
BOX_THRESHOLD = 0.30
TXT_THRESHOLD = 0.25
MANUAL_PROMPT = "green hydroflask water bottle"
AUTO_SELECT = True

if AUTO_SELECT:
    image_source, image = load_image(FIRST_FRAME)
    boxes, logits, phrases = predict(
        model=dino,
        image=image,
        caption=DINO_CAPTION,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TXT_THRESHOLD,
    )
    h, w = image_source.shape[:2]
    center = torch.tensor([0.5, 0.5], device=boxes.device)
    scores = []
    for i, (box, logit, phrase) in enumerate(zip(boxes, logits, phrases)):
        dist = torch.linalg.norm(box[:2] - center).item()
        scores.append((float(logit) - 0.5 * dist, i, str(phrase), float(logit)))
    assert scores, "Grounding DINO found no candidate objects. Set AUTO_SELECT=False and use MANUAL_PROMPT."
    _, best_i, target_object, confidence = max(scores, key=lambda x: x[0])
else:
    target_object, confidence = MANUAL_PROMPT, 1.0

sam_prompt = f"green {target_object} water bottle" if "green" not in target_object.lower() else target_object
print(f"DINO target: {target_object!r} confidence={confidence:.3f}")
print(f"SAM prompt: {sam_prompt!r}")

pip reports transformers==4.41.2
python -c import reports transformers==4.41.2
transformers file: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
BertModel has get_head_mask (type-level): 1


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


  https://huggingface.co/ShilongLiu/GroundingDINO/resolve/main/groundingdino_swint_ogc.pth
Saved: /content/groundingdino_swint_ogc.pth (694.0 MB)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1052: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/groundingdino/models/GroundingDINO/transformer.py:862: FutureWarning: `torch.cuda.amp.autocast(arg

DINO target: 'bottle container' confidence=0.423
SAM prompt: 'green bottle container water bottle'


## Stage 3 — SAM 3 Segmentation With Real Masks

v3 saves three artifacts:

- original RGB frames in `FRAMES_KEEP`
- binary mask PNGs in `MASKS_DIR`
- RGBA training PNGs in `FRAMES_RGBA`

The alpha channel is the important part. It gives the 2DGS camera loader a real foreground mask instead of forcing TSDF to guess from white pixels.

In [ ]:
# Install SAM 3
os.chdir("/content")
if not os.path.exists("/content/sam3"):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/facebookresearch/sam3.git"], check=True)
os.chdir("/content/sam3")
!pip install -q -e .
print("SAM 3 ready")

In [ ]:
import cv2

if restore_checkpoint(DRIVE_MASKS, MASKS_DIR, "SAM masks") and restore_checkpoint(DRIVE_RGBA, FRAMES_RGBA, "RGBA frames"):
    print("Using saved SAM outputs")
else:
    reset_dir(SAM_INPUT)
    reset_dir(MASKS_DIR)
    reset_dir(FRAMES_RGBA)

    orig_frames = sorted(os.listdir(FRAMES_KEEP))
    for i, fname in enumerate(orig_frames):
        shutil.copy2(f"{FRAMES_KEEP}/{fname}", f"{SAM_INPUT}/{i}.jpg")

    prompt_candidates = [
        sam_prompt,
        f"{target_object} on a wooden stool",
        "green hydroflask water bottle",
    ]

    def run_sam3(prompt):
        from sam3.model_builder import build_sam3_video_predictor
        pred = build_sam3_video_predictor()
        resp = pred.handle_request(dict(type="start_session", resource_path=SAM_INPUT))
        sid = resp["session_id"]
        pred.handle_request(dict(type="add_prompt", session_id=sid, frame_index=0, text=prompt))
        results, skipped = [], 0
        for r in pred.propagate_in_video(sid):
            masks = r["outputs"]["out_binary_masks"]
            results.append((r["frame_index"], masks))
            if len(masks) == 0:
                skipped += 1
        pred.close_session(sid)
        return results, skipped

    chosen_prompt = None
    for prompt in prompt_candidates:
        print(f"Trying SAM prompt: {prompt!r}")
        results, skipped = run_sam3(prompt)
        print(f"  masked {len(results) - skipped}/{len(results)} frames")
        if skipped == 0:
            chosen_prompt = prompt
            break

    assert chosen_prompt is not None, "SAM did not mask every frame. Inspect prompts/masks before continuing."

    for idx, masks in sorted(results, key=lambda x: x[0]):
        src_name = orig_frames[idx]
        stem = Path(src_name).stem
        rgb = cv2.imread(f"{FRAMES_KEEP}/{src_name}", cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        mask = np.asarray(masks[0]).astype(bool)

        # Mild cleanup removes isolated speckles without changing the object silhouette much.
        mask_u8 = (mask.astype(np.uint8) * 255)
        kernel = np.ones((3, 3), np.uint8)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)

        rgba = np.dstack([rgb, mask_u8])
        Image.fromarray(mask_u8).save(f"{MASKS_DIR}/{stem}.png")
        Image.fromarray(rgba).save(f"{FRAMES_RGBA}/{stem}.png")

    copytree_fresh(MASKS_DIR, DRIVE_MASKS)
    copytree_fresh(FRAMES_RGBA, DRIVE_RGBA)
    print(f"Saved masks and RGBA frames using prompt: {chosen_prompt!r}")

mask_files = assert_nonempty_dir(MASKS_DIR, "SAM masks")
rgba_files = assert_nonempty_dir(FRAMES_RGBA, "RGBA frames")
print(f"Masks: {len(mask_files)}  RGBA frames: {len(rgba_files)}")
print("Sample RGBA mode:", Image.open(f"{FRAMES_RGBA}/{rgba_files[0]}").mode)

In [ ]:
# Visual sanity check for masks and alpha frames
import matplotlib.pyplot as plt

frames = sorted(os.listdir(FRAMES_KEEP))
indices = [0, len(frames)//3, 2*len(frames)//3, len(frames)-1]
fig, axes = plt.subplots(len(indices), 3, figsize=(12, 4 * len(indices)))

for row, idx in enumerate(indices):
    stem = Path(frames[idx]).stem
    rgb = Image.open(f"{FRAMES_KEEP}/{frames[idx]}")
    mask = Image.open(f"{MASKS_DIR}/{stem}.png")
    rgba = Image.open(f"{FRAMES_RGBA}/{stem}.png")
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"RGB {frames[idx]}")
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("SAM mask")
    axes[row, 2].imshow(rgba)
    axes[row, 2].set_title("RGBA training image")
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

## Stage 4 — VGGT Camera Poses, COLMAP Format

VGGT runs on original RGB frames for better pose estimation. The exported COLMAP metadata is written at original image resolution, but `images.txt` points to the RGBA `.png` filenames that 2DGS will train on.

In [ ]:
# Install VGGT
!pip install -q git+https://github.com/facebookresearch/vggt.git
print("VGGT installed")

In [ ]:
if restore_checkpoint(DRIVE_COLMAP, COLMAP_OUT, "VGGT/COLMAP dataset"):
    print("Using saved COLMAP dataset")
else:
    import torch
    from scipy.spatial.transform import Rotation
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri

    reset_dir(COLMAP_OUT)
    os.makedirs(f"{COLMAP_OUT}/images", exist_ok=True)
    os.makedirs(f"{COLMAP_OUT}/sparse/0", exist_ok=True)

    # 2DGS training images are RGBA PNGs; COLMAP image names must match these files.
    for png in sorted(glob.glob(f"{FRAMES_RGBA}/*.png")):
        shutil.copy2(png, f"{COLMAP_OUT}/images/{Path(png).name}")

    rgb_frame_paths = sorted(glob.glob(f"{FRAMES_KEEP}/*.jpg"))
    sample = Image.open(rgb_frame_paths[0])
    W_orig, H_orig = sample.size
    print(f"Original image size: {W_orig} x {H_orig}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = VGGT.from_pretrained("facebook/VGGT-1B").to(device).eval()
    images = load_and_preprocess_images(rgb_frame_paths).to(device)
    print("VGGT input:", tuple(images.shape))

    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
        predictions = model(images)

    extrinsics, intrinsics = pose_encoding_to_extri_intri(predictions["pose_enc"], image_size_hw=(518, 518))
    extrinsics = extrinsics[0].cpu().float().numpy()
    intrinsics = intrinsics[0].cpu().float().numpy()

    scale_x = W_orig / 518.0
    scale_y = H_orig / 518.0

    with open(f"{COLMAP_OUT}/sparse/0/cameras.txt", "w") as cf:
        cf.write("# Camera list\n")
        for cam_id, K in enumerate(intrinsics, start=1):
            fx = float(K[0, 0]) * scale_x
            fy = float(K[1, 1]) * scale_y
            cx = float(K[0, 2]) * scale_x
            cy = float(K[1, 2]) * scale_y
            cf.write(f"{cam_id} PINHOLE {W_orig} {H_orig} {fx:.6f} {fy:.6f} {cx:.6f} {cy:.6f}\n")

    with open(f"{COLMAP_OUT}/sparse/0/images.txt", "w") as imf:
        imf.write("# Image list\n")
        for img_id, (rgb_path, E) in enumerate(zip(rgb_frame_paths, extrinsics), start=1):
            R = E[:3, :3]
            t = E[:3, 3]
            qw, qx, qy, qz = Rotation.from_matrix(R).as_quat()[[3, 0, 1, 2]]
            png_name = Path(rgb_path).with_suffix(".png").name
            imf.write(
                f"{img_id} {qw:.9f} {qx:.9f} {qy:.9f} {qz:.9f} "
                f"{t[0]:.9f} {t[1]:.9f} {t[2]:.9f} {img_id} {png_name}\n\n"
            )

    pts3d = predictions["world_points"].cpu().float().numpy().reshape(-1, 3)
    conf = predictions["world_points_conf"].cpu().float().numpy().reshape(-1)
    top_idx = np.argsort(conf)[-20000:]
    with open(f"{COLMAP_OUT}/sparse/0/points3D.txt", "w") as ptf:
        ptf.write("# Point3D list\n")
        for pt_id, xyz in enumerate(pts3d[top_idx], start=1):
            ptf.write(f"{pt_id} {xyz[0]:.6f} {xyz[1]:.6f} {xyz[2]:.6f} 128 128 128 0.0\n")

    copytree_fresh(COLMAP_OUT, DRIVE_COLMAP)
    print("Saved VGGT/COLMAP dataset")

print("COLMAP images:", len(os.listdir(f"{COLMAP_OUT}/images")))
print("Sparse files:", sorted(os.listdir(f"{COLMAP_OUT}/sparse/0")))
assert Image.open(sorted(glob.glob(f"{COLMAP_OUT}/images/*.png"))[0]).mode == "RGBA"

## Stage 5 — 2DGS Training

Train from the VGGT COLMAP dataset whose images are RGBA PNGs. After training, v3 checks that the dataset still contains alpha images; if 2DGS ever regresses to ignoring alpha, stop here and patch the loader before TSDF.

In [ ]:
# Install 2DGS and CUDA extensions
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0"
os.environ["MAX_JOBS"] = "4"
os.environ["FORCE_CUDA"] = "1"

os.chdir("/content")
if not os.path.exists("/content/2d-gaussian-splatting"):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/hbb1/2d-gaussian-splatting"], check=True)
os.chdir("/content/2d-gaussian-splatting")

!pip install -q plyfile tqdm
!pip install -q submodules/diff-surfel-rasterization
# Colab/Python versions sometimes fail on the vendored simple-knn; this upstream build is more reliable.
!pip install -q git+https://gitlab.inria.fr/bkerbl/simple-knn.git
print("2DGS ready")

In [ ]:
# Train 2DGS on RGBA PNGs
os.chdir("/content/2d-gaussian-splatting")

sample_image = sorted(glob.glob(f"{COLMAP_OUT}/images/*.png"))[0]
assert Image.open(sample_image).mode == "RGBA", "Training images must preserve alpha masks"

if os.path.exists(OUTPUT_2DGS):
    shutil.rmtree(OUTPUT_2DGS)

t0 = time.time()
subprocess.run([
    "python", "train.py",
    "-s", COLMAP_OUT,
    "-m", OUTPUT_2DGS,
    "--iterations", str(TRAIN_ITERS),
    "--save_iterations", str(TRAIN_ITERS),
], check=True)
print(f"Stage 5 wall time: {time.time() - t0:.1f}s")

copytree_fresh(OUTPUT_2DGS, DRIVE_2DGS)
print(f"Checkpointed 2DGS output -> {DRIVE_2DGS}")

## Stage 6 — TSDF Mesh Extraction

This is the v3 critical path. The previous white-background workaround is intentionally removed. If RGBA masks are loaded correctly, 2DGS should populate `gt_alpha_mask`, and the existing TSDF mask branch should suppress background pixels.

In [ ]:
# Mesh extraction with real alpha masks
!pip install -q trimesh open3d mediapy

os.chdir("/content/2d-gaussian-splatting")
reset_dir(OUTPUT_DIR)

t0 = time.time()
subprocess.run([
    "python", "render.py",
    "-m", OUTPUT_2DGS,
    "--skip_train",
    "--skip_test",
    "--depth_ratio", "1",
], check=True)
print(f"Stage 6 wall time: {time.time() - t0:.1f}s")

MESH_PLY = f"{OUTPUT_2DGS}/train/ours_{TRAIN_ITERS}/fuse_post.ply"
RAW_MESH_PLY = f"{OUTPUT_2DGS}/train/ours_{TRAIN_ITERS}/fuse.ply"
assert os.path.exists(MESH_PLY), f"TSDF mesh not found: {MESH_PLY}"
print("TSDF mesh:", MESH_PLY, os.path.getsize(MESH_PLY) / 1e6, "MB")

In [ ]:
# Mesh quality gate: fail fast on shattered geometry
import open3d as o3d
import trimesh

mesh_o3d = o3d.io.read_triangle_mesh(MESH_PLY)
mesh_o3d.remove_duplicated_vertices()
mesh_o3d.remove_degenerate_triangles()
mesh_o3d.remove_duplicated_triangles()
mesh_o3d.remove_non_manifold_edges()
mesh_o3d.compute_vertex_normals()

triangle_clusters, cluster_n_triangles, cluster_area = mesh_o3d.cluster_connected_triangles()
cluster_n_triangles = np.asarray(cluster_n_triangles)
cluster_area = np.asarray(cluster_area)
num_clusters = len(cluster_n_triangles)
main_ratio = float(cluster_n_triangles.max() / max(cluster_n_triangles.sum(), 1))

print(f"Connected components: {num_clusters:,}")
print(f"Largest component triangle ratio: {main_ratio:.3f}")
print(f"Vertices: {len(mesh_o3d.vertices):,}  Triangles: {len(mesh_o3d.triangles):,}")

# v2 shattered into tens of thousands of components. A good object mesh should not.
MESH_OK = (num_clusters < 500) and (main_ratio > 0.75)
print("MESH_OK:", MESH_OK)

if MESH_OK:
    accepted_ply = f"{OUTPUT_DIR}/model_tsdf_clean.ply"
    o3d.io.write_triangle_mesh(accepted_ply, mesh_o3d)
else:
    print("TSDF still looks fragmented. Do not use this as final unless visual inspection says otherwise.")

## Stage 7 — Export

Only export the v3 TSDF mesh if it passed the component quality gate. Otherwise, explicitly use the known clean v1 SuGaR mesh as the fallback deliverable and keep the failed v3 mesh for analysis.

In [ ]:
# Export GLB/STL/OBJ without mixing outputs from failed attempts
reset_dir(OUTPUT_DIR)

V1_SUGAR_GLOB = f"{DRIVE_ROOT}/truerender_sugar_output/refined_mesh/**/*.obj"
V1_CANDIDATES = sorted(glob.glob(V1_SUGAR_GLOB, recursive=True))

if MESH_OK:
    source_mesh_path = MESH_PLY
    source_label = "v3_tsdf"
elif V1_CANDIDATES:
    source_mesh_path = V1_CANDIDATES[-1]
    source_label = "v1_sugar_fallback"
else:
    raise RuntimeError("v3 TSDF failed and no v1 SuGaR fallback mesh was found on Drive")

print(f"Exporting from {source_label}: {source_mesh_path}")
mesh = trimesh.load(source_mesh_path, force="mesh", process=True)

# Keep a reasonable AR-friendly copy while preserving shape. STL/OBJ can still be high-res if needed.
if len(mesh.faces) > 200_000:
    try:
        mesh_export = mesh.simplify_quadric_decimation(face_count=200_000)
        print(f"Decimated for export: {len(mesh.faces):,} -> {len(mesh_export.faces):,} faces")
    except Exception as exc:
        print("Decimation failed; exporting original mesh:", exc)
        mesh_export = mesh
else:
    mesh_export = mesh

mesh_export.export(f"{OUTPUT_DIR}/model.glb")
mesh_export.export(f"{OUTPUT_DIR}/model.obj")
mesh_export.export(f"{OUTPUT_DIR}/model.stl")

with open(f"{OUTPUT_DIR}/source_manifest.json", "w") as f:
    json.dump({
        "source_label": source_label,
        "source_mesh_path": source_mesh_path,
        "mesh_ok": bool(MESH_OK),
        "num_clusters": int(num_clusters),
        "largest_component_ratio": main_ratio,
        "faces_exported": int(len(mesh_export.faces)),
    }, f, indent=2)

copytree_fresh(OUTPUT_DIR, DRIVE_FINAL)
print("Final exports:", sorted(os.listdir(OUTPUT_DIR)))
print(f"Saved to Drive: {DRIVE_FINAL}")

## Notes For Debugging v3

If TSDF still shatters after this version, inspect these in order:

1. Confirm every file in `COLMAP_OUT/images` is `RGBA`, not `RGB`.
2. Confirm 2DGS did not convert RGBA to RGB inside its dataset reader before camera construction.
3. Add a temporary print in `utils/mesh_utils.py` to count cameras where `gt_alpha_mask is not None`.
4. Verify VGGT intrinsics are scaled to the original frame size in `cameras.txt`.
5. If all of the above are correct and TSDF still fails, use v1 SuGaR for the deliverable and report v3 as a faster reconstruction path whose meshing stage needs further work.